In [1]:
!pip install -U torchvision

In [2]:
!pip install torchmetrics

In [3]:
!pip install segmentation_models_pytorch

In [4]:
!pip install torchinfo

In [5]:
!pip install albumentations

In [1]:
import torch
import torch.nn as nn
import torchvision

In [2]:
import os
import numpy as np
import cv2
from glob import glob
import random

In [3]:
import albumentations as album

In [4]:
import matplotlib.pyplot as plt

In [5]:
import warnings
warnings.filterwarnings("ignore")

import os, cv2, glob
import zipfile
import requests
import numpy as np
import pandas as pd
import random, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
from glob import glob
import torch
import torch.nn as nn
import albumentations as album
import torchmetrics
import torch.nn.functional as F
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F
import PIL
from PIL import Image
from tqdm import tqdm
from segmentation_models_pytorch.losses import DiceLoss
from torchinfo import summary
from torch.utils.data import DataLoader
from dataclasses import dataclass
from torch.optim.lr_scheduler import StepLR, OneCycleLR
from sklearn.model_selection import train_test_split

In [6]:
def conv_block(inp_filt, out_filt):
    conv = nn.Sequential(nn.Conv2d(inp_filt, out_filt, 3, padding=1),
                         nn.BatchNorm2d(out_filt),
                         nn.ReLU(inplace=True),
                         #nn.Dropout2d(0.003)
                         )
    return conv

In [7]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(nn.Conv2d(in_planes, in_planes // 16, 1, bias=False),
                               nn.ReLU(),
                               nn.Conv2d(in_planes // 16, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.avg_pool(x)
        avg_out = self.fc(avg_out)

        max_out = self.max_pool(x)
        max_out = self.fc(max_out)

        out = avg_out + max_out
        out = self.sigmoid(out)
        out = out * x
        return out

In [8]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.conv1(out)
        out = self.sigmoid(out)
        out = out * x
        return out


In [9]:
def cbam(x):
    in_planes = x.shape[1]
    ca = ChannelAttention(in_planes).to('cuda')
    sa = SpatialAttention(kernel_size=3).to('cuda')
    x = ca(x)
    x = sa(x)
    return x

In [10]:
class conv2blk(nn.Module):
    def __init__(self, inp_filt, out_filt=None):
        super().__init__()
        if not out_filt:
            out_filt = inp_filt

        self.conv = nn.Sequential(
            conv_block(inp_filt*2, inp_filt),
            conv_block(inp_filt, out_filt),
        )

    def forward(self, x):
        x = self.conv(x)
        x = cbam(x)
        return x

In [11]:
class decoder_blk(nn.Module):
    def __init__(self, inp_filt):
        super().__init__()
        self.conv = conv2blk(inp_filt)
        self.x2x2cc = nn.ConvTranspose2d(inp_filt, int(inp_filt/2), (2,2), 2)

    def forward(self, x):
        x = self.conv(x)
        x = self.x2x2cc(x)
        return x

In [12]:
backbone = torch.hub.load("pytorch/vision", "resnet50", weights="ResNet50_Weights.IMAGENET1K_V1")

Using cache found in /home/ec2-user/.cache/torch/hub/pytorch_vision_main


In [13]:
backbone = backbone.to('cuda')

In [14]:
summary(backbone, input_size=(1,3,224,224), device='cpu')

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 256, 56, 56]          --
│    └─Bottleneck: 2-1                   [1, 256, 56, 56]          --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           4,096
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-6                    [1, 64, 56, 56]           --
│ 

In [15]:
class Bro_RESUNET_CBAM(nn.Module):
    def __init__(self, model):
        super().__init__()
        d_f = [2048, 1024, 512, 256, 64]
        self.layer_0 = nn.Sequential(
            model.conv1, #x/2
            model.bn1,
            model.relu,
            model.maxpool #x/4, 64
        )
        self.layer_1 = model.layer1 #x/4, 256
        self.layer_2 = model.layer2 #x/8, 512
        self.layer_3 = model.layer3 #x/16, 1024
        self.layer_4_bneck = model.layer4 #x/32, 2048

        self.bneck_x_2x = nn.ConvTranspose2d(d_f[0], d_f[1], (2,2), 2) #x/16, 2048, 1024

        self.dec_blk_layer_3 = decoder_blk(d_f[1]) #2048, 1024, 512
        self.dec_blk_layer_2 = decoder_blk(d_f[2]) #1024, 512, 256
        self.dec_blk_layer_1 = conv2blk(d_f[3], d_f[4])  #512, 256, 64

        self.final_conv = nn.Sequential(
            nn.ConvTranspose2d(d_f[4]*2, d_f[4], (2,2), 2), #112
            nn.ConvTranspose2d(d_f[4], 1, (2,2), 2), #224
            #nn.Sigmoid() #we want to use BCELosswithLogits /Cross Entropy so we want to be flexible here
            )

    def forward(self, x):
        s1 = self.layer_0(x)  #224, 56
        s2 = self.layer_1(s1) #56, 56
        s3 = self.layer_2(s2) #56, 28
        s4 = self.layer_3(s3) #28, 14

        out = self.layer_4_bneck(s4) #14, 7
        out = self.bneck_x_2x(out) #7, 14

        out = torch.cat((s4, out), 1)
        out = self.dec_blk_layer_3(out) #14, 28
        out = torch.cat((s3, out), 1)
        out = self.dec_blk_layer_2(out) #28, 56
        out = torch.cat((s2, out), 1)
        out = self.dec_blk_layer_1(out) #56, 56
        out = torch.cat((s1, out), 1)
        out = self.final_conv(out) #56, 112, 224
        sig_out = nn.Sigmoid()(out)

        return out, sig_out

In [16]:
model = Bro_RESUNET_CBAM(backbone)

In [17]:
model = model.to('cuda')

In [18]:
summary(model, input_size=(1,3,512,512)) #, device='cpu')

Layer (type:depth-idx)                        Output Shape              Param #
Bro_RESUNET_CBAM                              [1, 1, 512, 512]          --
├─Sequential: 1-1                             [1, 64, 128, 128]         --
│    └─Conv2d: 2-1                            [1, 64, 256, 256]         9,408
│    └─BatchNorm2d: 2-2                       [1, 64, 256, 256]         128
│    └─ReLU: 2-3                              [1, 64, 256, 256]         --
│    └─MaxPool2d: 2-4                         [1, 64, 128, 128]         --
├─Sequential: 1-2                             [1, 256, 128, 128]        --
│    └─Bottleneck: 2-5                        [1, 256, 128, 128]        --
│    │    └─Conv2d: 3-1                       [1, 64, 128, 128]         4,096
│    │    └─BatchNorm2d: 3-2                  [1, 64, 128, 128]         128
│    │    └─ReLU: 3-3                         [1, 64, 128, 128]         --
│    │    └─Conv2d: 3-4                       [1, 64, 128, 128]         36,864
│    │  

In [19]:
inp = torch.ones(3, 3, 512, 512)
inp = inp.to('cuda')
inp.shape

torch.Size([3, 3, 512, 512])

In [20]:
out, sig_out = model(inp)
out.shape, sig_out.shape

(torch.Size([3, 1, 512, 512]), torch.Size([3, 1, 512, 512]))

In [21]:
out, sig_out

(tensor([[[[ 2.3372e+00,  2.5668e+00, -4.8092e-03,  ...,  5.7576e+00,
            -9.9121e-01,  1.2898e+00],
           [ 6.3952e-01, -5.1894e-01,  8.1205e-01,  ..., -2.7474e+00,
             4.4454e-01, -2.2128e+00],
           [ 7.4198e-01,  3.6518e+00,  1.9361e+00,  ...,  6.4067e-01,
             2.2948e+00,  1.8376e+00],
           ...,
           [-8.9094e-01, -6.0141e-02,  4.1208e-01,  ..., -2.7903e+00,
            -1.1852e-01, -1.5152e+00],
           [ 1.6308e+00,  2.7595e+00,  1.4044e+00,  ...,  2.2551e+00,
             2.4526e+00,  1.0932e-01],
           [-2.6281e+00,  1.0438e+00, -1.3283e+00,  ...,  2.7684e+00,
            -8.4049e-01, -8.6644e-01]]],
 
 
         [[[ 2.3372e+00,  2.5668e+00, -4.8092e-03,  ...,  5.7576e+00,
            -9.9121e-01,  1.2898e+00],
           [ 6.3952e-01, -5.1894e-01,  8.1205e-01,  ..., -2.7474e+00,
             4.4454e-01, -2.2128e+00],
           [ 7.4198e-01,  3.6518e+00,  1.9361e+00,  ...,  6.4067e-01,
             2.2948e+00,  1.8376e+00

In [22]:
model

Bro_RESUNET_CBAM(
  (layer_0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer_1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256

In [23]:
def get_training_augmentation():
    return  album.Compose([
        album.Resize(
            height=height,
            width=width,
            always_apply=True,
            interpolation=cv2.INTER_LINEAR,
        ),
        album.HorizontalFlip(p=0.5),
        album.RandomBrightnessContrast(p=0.2),
        album.Rotate(limit=25),
        album.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
            max_pixel_value=1.0
        )
    ])

def get_validation_augmentation():
    return album.Compose([
        album.Resize(
            height=height,
            width=width,
            always_apply=True,
            interpolation=cv2.INTER_LINEAR,
        ),
        album.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
            max_pixel_value=1.0
        )
    ])

In [24]:
# Load the trained model.
#trained_model = DinoUNetResNet50(num_classes=DatasetConfig.NUM_CLASSES).to(TrainingConfig.DEVICE)
trained_model = Bro_RESUNET_CBAM(backbone)
ckpt = torch.load('outputs/best_iou.pt')
trained_model.load_state_dict(ckpt)
trained_model = trained_model.to('cuda')

In [25]:
def generate_predictions(model, images):
    with torch.no_grad():
        out, sig_out = model(images)
        #predictions = nn.Sigmoid()(predictions)

    return out, sig_out

In [26]:
height = 512
width = 512

In [27]:
dataset_path = "test-images"
save_path = os.path.join(dataset_path, "outputs")

In [28]:
save_path

'test-images/outputs'

In [29]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [30]:
create_dir(save_path)

In [31]:
test_x = sorted(glob(os.path.join(dataset_path, "images", "*")))
print("Test Images: ", len(test_x))

Test Images:  10


In [32]:
test_x[0]

'test-images/images/heybike-DrQN1IUokV8-unsplash.jpg'

In [33]:
trained_model.eval()
augment = get_validation_augmentation()

In [42]:
for x in tqdm(test_x):
    name = x.split("/")[-1]
    image = cv2.imread(x, cv2.IMREAD_COLOR)
    img_x = image
    h, w, _ = image.shape
    image = augment(image=image)['image']
    image = np.expand_dims(image, axis=0)
    image = image.transpose(0, 3, 1, 2)
    image = torch.Tensor(image).to('cuda')
    _, p = generate_predictions(trained_model, image)
    p = p.squeeze(0)
    p = p.detach().cpu().numpy()
    p = p.transpose(1, 2, 0)
    p = cv2.resize(p, (w, h))
    p = np.expand_dims(p, axis=-1)
    p = p > 0.5
    foreground_mask = p
    background_mask = 1 - p
    
    foreground_photo = img_x * foreground_mask
    background_photo = img_x * background_mask
    background_photo = cv2.blur(background_photo, (21, 21))
    
    final_photo = foreground_photo + background_photo
    
    line = np.ones((h, 10, 3)) * 255
    cat_img = np.concatenate([img_x, line, final_photo], axis=1)
    
    cv2.imwrite(f"{save_path}/{name}", cat_img)

100%|██████████| 10/10 [00:05<00:00,  1.78it/s]
